In [1]:
import pandas as pd
import numpy as np 
#load raw csv v2
df = pd.read_csv("./datasets/raw/recruitment_funnel_v2.csv")

## Task 1 : Load and Inspect

In [2]:
print('shape:', df.shape)
display(df.info())
print("/n Missing Values:\n",df.isna().sum())
print('/nRaw source unique count:',df['source'].nunique())
print('/nRaw department unique count:',df['department'].nunique())

shape: (1515, 16)
<class 'pandas.DataFrame'>
RangeIndex: 1515 entries, 0 to 1514
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   applicant_id       1515 non-null   int64  
 1   name               1515 non-null   str    
 2   age                1515 non-null   int64  
 3   gender             1515 non-null   str    
 4   experience_years   1515 non-null   float64
 5   source             1515 non-null   str    
 6   job_role           1515 non-null   str    
 7   department         1515 non-null   str    
 8   recruiter          1515 non-null   str    
 9   applied_date       1515 non-null   str    
 10  screened_date      862 non-null    str    
 11  interviewed_date   463 non-null    str    
 12  offered_date       156 non-null    str    
 13  hired_date         122 non-null    str    
 14  final_status       1515 non-null   str    
 15  rejected_at_stage  1221 non-null   str    
dtypes: float64(1), in

None

/n Missing Values:
 applicant_id            0
name                    0
age                     0
gender                  0
experience_years        0
source                  0
job_role                0
department              0
recruiter               0
applied_date            0
screened_date         653
interviewed_date     1052
offered_date         1359
hired_date           1393
final_status            0
rejected_at_stage     294
dtype: int64
/nRaw source unique count: 19
/nRaw department unique count: 14


## Task 2 Standardize Text & Source Mapping

In [3]:
# Keep raw source for AI #1 checks
df['source_raw'] = df['source']

# Strip whitespace and normalize case for mapping
df['name'] = df['name'].str.strip()
df['department'] = df['department'].str.strip()

# Normalize source for robust mapping (strip + lowercase)
df['source_clean_key'] = df['source'].str.strip().str.lower()

# Define dictionary mapping lowercase messy keys to 6 canonical sources
source_mapping = {
    'linkedin': 'LinkedIn',
    'linked in': 'LinkedIn',
    'referral': 'Referral',
    'indeed': 'Indeed',
    'company website': 'Company Website',
    'company website ': 'Company Website',
    'job fair': 'Job Fair',
    'jobfair': 'Job Fair',
    'naukri': 'Naukri'
}

df['source'] = df['source_clean_key'].map(source_mapping)
df = df.drop(columns=['source_clean_key'])

# Assertions
assert df['source'].nunique() == 6, f"Expected 6 unique sources, got {df['source'].nunique()}"
assert df['department'].nunique() == 7, f"Expected 7 unique departments, got {df['department'].nunique()}"
print("Text standardization and mapping successful! Unique sources: 6, Unique departments: 7.")

Text standardization and mapping successful! Unique sources: 6, Unique departments: 7.


## Task 3 : Handle Dates & Check Sequence Integrity

In [4]:
date_cols = ['applied_date', 'screened_date', 'interviewed_date', 'offered_date', 'hired_date']
for col in date_cols:
    df[col]=pd.to_datetime(df[col],errors ='coerce' )
# Check that every stage date is on or after the previous stage date
violations = ( 
    (df['screened_date'].notna() & (df['screened_date']< df['applied_date'])).sum() + 
    (df['interviewed_date'].notna() & (df['interviewed_date'] < df['screened_date'])).sum() +
    (df['offered_date'].notna() & (df['offered_date'] < df['interviewed_date'])).sum() + 
    (df['hired_date'].notna() & (df['hired_date'] < df['offered_date'])).sum()
       
)
print (f'Stage order violations found: {violations}') # expecting = 0
assert violations == 0
    

Stage order violations found: 0


## Task 4: Invalid Ages

In [5]:
invalid_age_mask = (df['age'] < 18) | (df['age'] > 65)
invalid_count =invalid_age_mask.sum()
print(f"Invalid ages found: {invalid_count}") # expecting =12 
df.loc[invalid_age_mask,'age']=np.nan
assert df['age'].isna().sum() >= 12

Invalid ages found: 12


## Task 5: Deduplication (Survivorship Rule)

In [6]:
dup_mask = df.duplicated(subset=['applicant_id'], keep=False)
duplicates_df = df[dup_mask].copy()

# Identify conflicting source pairs among duplicates
conflicting_ids = []
for aid, group in duplicates_df.groupby('applicant_id'):
    if group['source'].nunique() > 1:
        conflicting_ids.append(aid)

conflicting_pairs_review = duplicates_df[duplicates_df['applicant_id'].isin(conflicting_ids)]
print(f"Flagged {len(conflicting_pairs_review)} conflicting source rows for review.")

# Stable deduplication: sort by original index with kind='stable'
df['original_order'] = df.index
df = df.sort_values(by='original_order', kind='stable')
df = df.drop_duplicates(subset=['applicant_id'], keep='first').copy()
df = df.drop(columns=['original_order'])

assert df['applicant_id'].nunique() == 1500
assert len(df) == 1500

Flagged 16 conflicting source rows for review.


## Task 6: Feature Engineering

In [7]:
# 1. Convert all stage columns to datetime first
date_cols = ['applied_date', 'screened_date', 'interviewed_date', 'offered_date', 'hired_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Now calculate stage day gaps safely
df['days_applied_to_screen'] = (df['screened_date'] - df['applied_date']).dt.days
df['days_screen_to_interview'] = (df['interviewed_date'] - df['screened_date']).dt.days
df['days_interview_to_offer'] = (df['offered_date'] - df['interviewed_date']).dt.days
df['days_offer_to_hire'] = (df['hired_date'] - df['offered_date']).dt.days

# 3. Total days to hire (hired applicants only)
df['total_days_to_hire'] = (df['hired_date'] - df['applied_date']).dt.days


#binary indicator
df['is_hired']= (df['final_status'] == "Hired").astype(int)
# Stage reached (0 = Applied, 1 = Screened, 2 = Interviewed, 3 = Offered, 4 = Hired)
df['stage_reached']= 0
df.loc[df['screened_date'].notna(),'stage_reached']= 1
df.loc[df['interviewed_date'].notna(),'stage_reached'] = 2
df.loc[df['offered_date'].notna(),'stage_reached'] = 3
df.loc[df['hired_date'].notna(),'stage_reached'] = 4
# Application year-month
df['applied_month'] = df['applied_date'].dt.to_period('M').astype(str)
print("Feature engineering completed successfully.")

Feature engineering completed successfully.


## Task 7: Wide to Long Transformation

In [8]:
# Melt the funnel stages
df_long = df.melt(
    id_vars=['applicant_id', 'name', 'gender', 'experience_years', 'source', 'department', 'job_role', 'recruiter', 'final_status', 'is_hired'],
    value_vars=['applied_date', 'screened_date', 'interviewed_date', 'offered_date', 'hired_date'],
    var_name='stage_col',
    value_name='stage_date'
).dropna(subset=['stage_date'])

# Map stage columns to clean stage names and ordering
stage_order_map = {
    'applied_date': '1. Applied',
    'screened_date': '2. Screened',
    'interviewed_date': '3. Interviewed',
    'offered_date': '4. Offered',
    'hired_date': '5. Hired'
}
df_long['stage'] = df_long['stage_col'].map(stage_order_map)
df_long = df_long.drop(columns=['stage_col']).sort_values(by=['applicant_id', 'stage']).reset_index(drop=True)

print("Long format shape:", df_long.shape)
display(df_long.head())

Long format shape: (3078, 12)


,applicant_id,name,gender,experience_years,source,department,job_role,recruiter,final_status,is_hired,stage_date,stage
0,1,Allison Hill,Male,0.0,Referral,Marketing,Sales Associate,David Lee,Rejected,0,2025-11-26,1. Applied
1,1,Allison Hill,Male,0.0,Referral,Marketing,Sales Associate,David Lee,Rejected,0,2025-11-29,2. Screened
2,2,Noah Rhodes,Female,5.1,LinkedIn,Finance,UX Designer,Aisha Khan,Rejected,0,2025-10-01,1. Applied
3,2,Noah Rhodes,Female,5.1,LinkedIn,Finance,UX Designer,Aisha Khan,Rejected,0,2025-10-03,2. Screened
4,2,Noah Rhodes,Female,5.1,LinkedIn,Finance,UX Designer,Aisha Khan,Rejected,0,2025-10-07,3. Interviewed


## Task 8: Quality Checks, Summary Table, and Export

In [9]:
# Final Assertions
assert df['applicant_id'].nunique() == 1500
assert df['age'].between(18, 65).sum() > 0
assert df['source'].nunique() == 6
assert violations == 0

# Before / After summary table
summary = {
    "Metric": ["Total Rows", "Unique Applicants", "Unique Sources", "Unique Departments", "Invalid Ages"],
    "Before Cleaning": [1515, 1500, 19, 14, 12],
    "After Cleaning": [len(df), df['applicant_id'].nunique(), df['source'].nunique(), df['department'].nunique(), df['age'].isna().sum()]
}
display(pd.DataFrame(summary))

# Export datasets
df.to_csv("cleaned_funnel_data.csv", index=False)
df_long.to_csv("funnel_long.csv", index=False)
print("Successfully exported cleaned_funnel_data.csv and funnel_long.csv!")

,Metric,Before Cleaning,After Cleaning
0,Total Rows,1515,1500
1,Unique Applicants,1500,1500
2,Unique Sources,19,6
3,Unique Departments,14,7
4,Invalid Ages,12,12


Successfully exported cleaned_funnel_data.csv and funnel_long.csv!


In [10]:
df = pd.read_csv("./datasets/raw/recruitment_funnel_v2.csv")

In [11]:
import os
os.makedirs("data/processed", exist_ok=True)

df.to_csv("data/processed/cleaned_funnel_data.csv", index=False)
df_long.to_csv("data/processed/funnel_long.csv", index=False)
conflicting_pairs_review.to_csv("data/processed/conflicting_source_pairs_review.csv", index=False)
print("All files exported successfully!")

All files exported successfully!
